# NEXUS Alpha: Colab run-all (fundamentals reproduction)

**Use: Runtime -> Run all.** One pass: about 2-4 hours on a standard 2-CPU runtime, about 1-2 hours on a runtime with more CPUs (Pro+ high-RAM, or an A100 runtime). No
GPU is needed: this path is pandas and LightGBM. A GPU runtime also works, and the summary records which hardware ran it.

What it does, in order (all in `scripts/colab/reproduce.py`):
1. checks that the staged data is on Drive (made on the PC with `python scripts/colab/stage_for_drive.py --go`);
2. copies it to Colab's local disk (the Drive mount is slow for thousands of small files);
3. layout check, then the fundamentals, nightly-runner and repository-hygiene tests;
4. rebuilds the daily ratio panel, runs the audit's 7 hard checks;
5. re-runs E22 and E23, rebuilds the Alpha158 features and re-runs E25 (the Qlib benchmark), and compares all three with the committed, pre-registered records (**REPRODUCED** or **MISMATCH**);
6. verifies the frozen `v3_fund` bundle file by file.

At the end it writes **one small file**, `COLAB_RESULTS.md`, to `MyDrive/antigravity_workspace/colab_results/<time>/` and
prints it. Copy that file's text back to your reviewer (Claude Code or Antigravity). It contains no data, no secrets and no
email address.

Never done here: the sealed 2023+ period, the forward test (`shadow/`), refitting a frozen model, committing anything.
No secret is needed.

The other notebook, `NEXUS_Alpha_Colab_Orchestrator.ipynb`, is the full v1 pipeline: it needs about 15 GB of private data
and is not meant for Run all.

In [ ]:
# Settings
DRIVE_ROOT = '/content/drive/MyDrive/antigravity_workspace'   # where stage_for_drive.py put the data
QUICK = False      # True: skip the ratio build, E22, E23 and E25 (a 5-minute check of everything else; reproduces nothing)
WITH_E25 = True    # False: skip the Qlib Alpha158 test (E25), which adds about 1-3 hours

In [ ]:
# 1. Drive and code (the repository is public; it is cloned to local disk, never written to Drive)
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
REPO = '/content/nexus-alpha'
if os.path.exists(REPO):
    subprocess.run(['git', '-C', REPO, 'pull', '-q', '--ff-only'], check=False)
else:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/harshatangirala/nexus-alpha', REPO], check=True)
os.chdir(REPO)
print('code at commit', subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip())

In [ ]:
# 2. Install the pinned packages this path needs (torch is not reinstalled). If the pins cannot install on this Python,
#    it falls back to current versions; the summary records the versions that actually ran.
!pip -q install -r requirements-colab-repro.txt || (echo "PINNED INSTALL FAILED: falling back to unpinned versions" && pip -q install $(grep -v '^#' requirements-colab-repro.txt | cut -d= -f1))

In [ ]:
# 3. Run everything. Each step prints as it goes; the summary prints at the end.
extra = ('--skip-experiments' if QUICK else '') + ('' if WITH_E25 else ' --skip-e25')
!python scripts/colab/reproduce.py --drive-root "{DRIVE_ROOT}" {extra}

## Next

Open `MyDrive/antigravity_workspace/colab_results/<latest>/COLAB_RESULTS.md` (also printed above) and paste its text to your
reviewer. If the verdict is `BLOCKED: data not on Drive`, wait until Google Drive for desktop finishes syncing on the PC
and run all again (finished steps are cheap to repeat).